# 02 - Validacao PyTorch

Recriacao da MLP com `nn.Linear` e comparacao das curvas com a implementacao NumPy.

In [1]:
import sys
from pathlib import Path

for src_path in [Path.cwd() / 'src', Path.cwd().parent / 'src', Path('/content/ap2-ia/src')]:
    if src_path.exists():
        sys.path.insert(0, str(src_path))
        break

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from medmnist import PathMNIST
from utils import set_seed

set_seed(42)

In [2]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 9))
    def forward(self, x):
        return self.net(x)

train_ds = PathMNIST(split='train', size=28, download=True)
x_np = train_ds.imgs.astype('float32') / 255.0
x_np = x_np.mean(axis=-1)  # RGB 28x28x3 -> grayscale 28x28, mesma entrada 784 da MLP NumPy
x = torch.tensor(x_np, dtype=torch.float32).flatten(1)
y = torch.tensor(train_ds.labels.reshape(-1), dtype=torch.long)
loader = DataLoader(TensorDataset(x, y), batch_size=128, shuffle=True)

model = TorchMLP()
opt = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
criterion = nn.CrossEntropyLoss()
history_torch = []

for epoch in range(20):
    losses = []
    for xb, yb in loader:
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    with torch.no_grad():
        pred = model(x[:5000]).argmax(1)
        acc = (pred == y[:5000]).float().mean().item()
    history_torch.append({'epoch': epoch + 1, 'loss': sum(losses) / len(losses), 'acc_sample': acc})
    print(history_torch[-1])

100%|██████████| 205615438/205615438 [00:17<00:00, 11723112.01it/s]


{'epoch': 1, 'loss': 2.136100895364176, 'acc_sample': 0.21580000221729279}
{'epoch': 2, 'loss': 2.0044104660099205, 'acc_sample': 0.18880000710487366}
{'epoch': 3, 'loss': 1.9491797512905165, 'acc_sample': 0.2547999918460846}
{'epoch': 4, 'loss': 1.849711961536245, 'acc_sample': 0.3197999894618988}
{'epoch': 5, 'loss': 1.7716503763063387, 'acc_sample': 0.296999990940094}
{'epoch': 6, 'loss': 1.7336094865406102, 'acc_sample': 0.23680000007152557}
{'epoch': 7, 'loss': 1.701813897795298, 'acc_sample': 0.2614000141620636}
{'epoch': 8, 'loss': 1.6993129825727507, 'acc_sample': 0.3653999865055084}
{'epoch': 9, 'loss': 1.6840835003690287, 'acc_sample': 0.34139999747276306}
{'epoch': 10, 'loss': 1.6673750840127468, 'acc_sample': 0.32280001044273376}
{'epoch': 11, 'loss': 1.6664399602873758, 'acc_sample': 0.34439998865127563}
{'epoch': 12, 'loss': 1.6465939769352025, 'acc_sample': 0.3898000121116638}
{'epoch': 13, 'loss': 1.6390645544637332, 'acc_sample': 0.32179999351501465}
{'epoch': 14, 'los

Plote aqui as curvas NumPy e PyTorch sobrepostas e registre se a diferenca final de acuracia ficou em ate 2 pontos percentuais.